In [ ]:
#Global Imports

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window


**Part 1: Data Loading & Setup**

Q1. Spark Session & Data Loading

In [94]:

spark = SparkSession.builder.getOrCreate()
df=spark.read.csv("/content/sales_data.csv",inferSchema=True,header=True)
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



Q2. Data Type Conversion


In [47]:
#Already in the required DataTypes!
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



**Part 2: Aggregate Functions**    
Q3. Overall Sales Metrics


In [51]:
#Total Sales
df.select(sum("order_amount").alias("total_sales")).show()

#Average Order Amounts:
df.select(avg("order_amount").alias("average_order_amount")).show()

#Max and Min Order Amounts:
df.select(max("order_amount").alias("max_order_amount"), min("order_amount").alias("min_order_amount")).show()

+-----------+
|total_sales|
+-----------+
|     359000|
+-----------+

+--------------------+
|average_order_amount|
+--------------------+
|             44875.0|
+--------------------+

+----------------+----------------+
|max_order_amount|min_order_amount|
+----------------+----------------+
|           80000|           20000|
+----------------+----------------+



Q4. Region-wise Analysis:    
For each region, calculate:

In [52]:
#Grouped by Region
grp_region=df.groupBy("region")


#Total Sales
grp_region.agg(sum("order_amount").alias("total_sales")).show()

#Average Sales
grp_region.agg(avg("order_amount").alias("average_sales")).show()

#Order Count
grp_region.agg(count("order_id").alias("order_count")).show()


+------+-----------+
|region|total_sales|
+------+-----------+
| South|     102000|
|  East|     102000|
|  West|      28000|
| North|     127000|
+------+-----------+

+------+------------------+
|region|     average_sales|
+------+------------------+
| South|           51000.0|
|  East|           51000.0|
|  West|           28000.0|
| North|42333.333333333336|
+------+------------------+

+------+-----------+
|region|order_count|
+------+-----------+
| South|          2|
|  East|          2|
|  West|          1|
| North|          3|
+------+-----------+



Q5. Customer Count:

In [57]:
#Distinct Customers
df.select(count_distinct("customer_id").alias("distinct_customers")).show()

+------------------+
|distinct_customers|
+------------------+
|                 4|
+------------------+



Q6. Product-wise Aggregation:

In [64]:
#Group by Products
grp_products= df.groupby("product")


#Collect List
grp_products.agg(collect_list("order_amount")).show()

#Collect Set
grp_products.agg(collect_set("order_amount")).show()

+-------+--------------------------+
|product|collect_list(order_amount)|
+-------+--------------------------+
| Laptop|      [75000, 80000, 72...|
| Mobile|      [30000, 28000, 32...|
| Tablet|            [20000, 22000]|
+-------+--------------------------+

+-------+-------------------------+
|product|collect_set(order_amount)|
+-------+-------------------------+
| Laptop|     [72000, 80000, 75...|
| Mobile|     [28000, 30000, 32...|
| Tablet|           [20000, 22000]|
+-------+-------------------------+



**Part 3: Window Functions-Ranking**                         
Q7. Regional Ranking:  
For Each Region    

In [69]:
#Window by Region
reg_window=Window.partitionBy("region").orderBy("order_amount")

#Row Number ON Order Amount
df.withColumn("order_row",row_number().over(reg_window)).show()

#Rank and DenseRank
df.withColumn("order_rank",rank().over(reg_window)) \
  .withColumn("order_den_rank",dense_rank().over(reg_window)).show()

+--------+-----------+------+-------+------------+----------+-------------------+---------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|order_row|
+--------+-----------+------+-------+------------+----------+-------------------+---------+
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|        1|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|        2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|        1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|        2|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|        3|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|        1|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|        2|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00

Q8. Top Orders per Region

In [70]:
#Top2 Orders per Region
df.withColumn("top2_per_region",rank().over(reg_window)).filter("top2_per_region<=2").show()

+--------+-----------+------+-------+------------+----------+-------------------+---------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|top2_per_region|
+--------+-----------+------+-------+------------+----------+-------------------+---------------+
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|              1|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|              2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|              1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|              2|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|              1|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|              2|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|              1|
+--------+----------

**Part 4: Window Functions – Analytical**     
 Q9. Order Comparison per Customer

In [75]:
#Window By Customer
cust_window=Window.partitionBy("customer_id").orderBy("order_date")

#Old Amount (lag)
df.withColumn("old_amount",lag("order_amount",1,None).over(cust_window)).show()

#Next Amount (lead)
df.withColumn("next_amount",lead("order_amount",1,None).over(cust_window)).show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|old_amount|
+--------+-----------+------+-------+------------+----------+-------------------+----------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      NULL|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|     75000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|     20000|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|      NULL|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|     30000|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      NULL|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|     80000|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-0

Q10. Running Total

In [79]:
#Cumulative Sum of Order Amount
df.withColumn("cum_sum",sum("order_amount").over(cust_window)).show()

+--------+-----------+------+-------+------------+----------+-------------------+-------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|cum_sum|
+--------+-----------+------+-------+------------+----------+-------------------+-------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|  75000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|  95000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00| 127000|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|  30000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00| 102000|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|  80000|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00| 102000|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|  28000|
+--------+

**Part 5: Window Functions – Aggregates**

Q11. Average Order per Customer

In [77]:
#Average Order per Customer
df.withColumn("avg_customer_order",avg("order_amount").over(cust_window)).show()

+--------+-----------+------+-------+------------+----------+-------------------+------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|avg_order_per_cust|
+--------+-----------+------+-------+------------+----------+-------------------+------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|           75000.0|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|           47500.0|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|42333.333333333336|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|           30000.0|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|           51000.0|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|           80000.0|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|          

Q12. Maximum Order per Region

In [80]:
#Max Order per Region
df.withColumn("max_region_order",max("order_amount").over(reg_window)).show()

+--------+-----------+------+-------+------------+----------+-------------------+----------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|max_region_order|
+--------+-----------+------+-------+------------+----------+-------------------+----------------+
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|           22000|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|           80000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|           20000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|           32000|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|           75000|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|           30000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|           72000|
|    1005|

**Part 6: Date & Timestamp Functions**

Q13. Date Components Extraction:

In [81]:
#Extracting Date Components
df.withColumn("year",year("order_date")) \
.withColumn("month",month("order_date")) \
.withColumn("day",dayofmonth("order_date")).show()


+--------+-----------+------+-------+------------+----------+-------------------+----+-----+---+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|year|month|day|
+--------+-----------+------+-------+------------+----------+-------------------+----+-----+---+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|2023|    1| 10|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|2023|    1| 12|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|2023|    2|  5|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|2023|    2| 20|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|2023|    3|  1|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|2023|    3| 15|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|2023|    3| 18|
|    1008|       C003|  East| 

Q14. Date Difference:


In [82]:
#Differnce from Order Date to Today
df.withColumn("days_since_order",datediff(current_date(),col("order_date"))).show()

+--------+-----------+------+-------+------------+----------+-------------------+----------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|days_since_order|
+--------+-----------+------+-------+------------+----------+-------------------+----------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|            1171|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|            1169|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|            1145|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|            1130|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|            1121|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|            1107|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|            1104|
|    1008|

Q15. Create new columns:

In [96]:
# Create New Column as Order year,month and week
df = df.withColumn("order_year",year("order_date")) \
.withColumn("order_month",month("order_date")) \
.withColumn("order_week",weekofyear("order_date"))
df.show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|order_year|order_month|order_week|
+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      2023|          1|         2|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|      2023|          1|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      2023|          2|         5|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      2023|          2|         8|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|      2023|          3|         9|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 

Q16. Date-based Filtering:


In [85]:
#Order Placed in March 2023
df.filter(  (year("order_date")==2023) & (month("order_date")==3)).show()


+--------+-----------+------+-------+------------+----------+-------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|
+--------+-----------+------+-------+------------+----------+-------------------+
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|
+--------+-----------+------+-------+------------+----------+-------------------+



**Part 7: Real-World ETL Scenarios**

Q17. Monthly Sales Trend:

In [99]:
#Monthly and Yearly Group Sales
df.groupBy("order_year","order_month").agg(sum("order_amount").alias("total_sales")).orderBy("order_month").show()

+----------+-----------+-----------+
|order_year|order_month|total_sales|
+----------+-----------+-----------+
|      2023|          1|     105000|
|      2023|          2|     100000|
|      2023|          3|     132000|
|      2023|          4|      22000|
+----------+-----------+-----------+



Q18. Customer Activity Analysis:

In [100]:
#Customer having more than 2 Orders
df.groupBy("customer_id").agg(count("order_id").alias("order_count")).filter("order_count>2").show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|       C001|          3|
+-----------+-----------+

